In [ ]:
# ZarAI E3 final: clean Roman Urdu conversion + fair DR-RAG baseline, scored on the unseen half. GPU T4, Internet on.
!pip -q install -U sentence-transformers rank_bm25 bitsandbytes accelerate

import json, glob, re, gc, random, unicodedata, numpy as np, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

chunks = [json.loads(l) for l in open(find("chunks.jsonl"), encoding="utf-8")]
qa = [json.loads(l) for l in open(find("qa.jsonl"), encoding="utf-8")]
qa = [r for r in qa if r["kb_chunk_ids"]]
ids = [c["chunk_id"] for c in chunks]
N = len(ids)
fact = lambda r: r["notes"].split(";")[0]
facts = sorted({fact(r) for r in qa})
random.Random(42).shuffle(facts)
dev_facts = set(facts[: len(facts) // 2])
test = [r for r in qa if fact(r) not in dev_facts]

name = "Qwen/Qwen2.5-14B-Instruct"
tokzr = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(
    name, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
def chat(msgs, max_new):
    x = tokzr.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**x, max_new_tokens=max_new, do_sample=False)
    return tokzr.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

URDU = re.compile(r"^[؀-ۿݐ-ݿﭐ-﷿ﹰ-\uFEFF0-9\s\-،؟?!.,:()/%+]+$")
def clean(text):
    line = text.splitlines()[0] if text else ""
    line = re.sub(r"^(Corrected|Note|Urdu)\s*:\s*", "", line, flags=re.I).strip()
    return line

# 1. Roman Urdu -> Urdu script: first line only, Urdu characters only, retry once if not clean
SHOTS = [("gandum ki fasal achhi hai", "گندم کی فصل اچھی ہے"), ("kungi ka hamla ho gaya", "کنگی کا حملہ ہو گیا"),
         ("telay ne patte kha liye", "تیلے نے پتے کھا لیے"), ("khaad aur pani waqt par dein", "کھاد اور پانی وقت پر دیں"),
         ("beej ki sharah barha dein", "بیج کی شرح بڑھا دیں"), ("kheton mein jari bootiyan hain", "کھیتوں میں جڑی بوٹیاں ہیں"),
         ("barani ilaqon mein barish kam hai", "بارانی علاقوں میں بارش کم ہے"), ("nehri pani mil raha hai", "نہری پانی مل رہا ہے"),
         ("urea ki bori kitne ki hai?", "یوریا کی بوری کتنے کی ہے؟"), ("katai ke baad toori banayein", "کٹائی کے بعد توڑی بنائیں"),
         ("zameen ka tajziya karwayein", "زمین کا تجزیہ کروائیں"), ("spray subah karein ya sham ko?", "سپرے صبح کریں یا شام کو؟"),
         ("dhaan ke baad kya lagayein?", "دھان کے بعد کیا لگائیں؟"), ("sitton mein dana kam hai", "سٹوں میں دانہ کم ہے"),
         ("deemak se kaise bachein?", "دیمک سے کیسے بچیں؟")]
SYS = ("Convert Roman Urdu into Urdu script. Keep the same words and order, even if a word looks wrong. "
       "Write English words such as variety names in Urdu letters. Output one line of Urdu script only: "
       "no notes, no corrections, no other language.")
def convert(text, strict=False):
    msgs = [{"role": "system", "content": SYS + (" Use only Urdu letters and digits." if strict else "")}]
    for a, b in SHOTS:
        msgs += [{"role": "user", "content": a}, {"role": "assistant", "content": b}]
    return clean(chat(msgs + [{"role": "user", "content": text}], 60))
translit, retried, still_bad = {}, 0, 0
for r in [x for x in qa if x["lang"] == "ur-Latn"]:
    t = convert(r["question"])
    if not URDU.match(t):
        retried += 1
        t = convert(r["question"], strict=True)
        still_bad += not URDU.match(t)
    translit[r["id"]] = t
print(f"conversions: {len(translit)}, retried {retried}, still not pure Urdu {still_bad}")

# 2. DR-RAG question index: Urdu only, no answers, retry chunks that give too few clean questions
QSYS = ("Read this Urdu wheat-farming passage. Write 6 short questions in Urdu script that a farmer could ask "
        "and that this passage answers. Each line must be one question ending with ؟. "
        "Do not write answers, numbers lists, or any other language.")
def questions(text, strict=False):
    msgs = [{"role": "system", "content": QSYS + (" Urdu letters only." if strict else "")}, {"role": "user", "content": text[:2500]}]
    out = []
    for line in chat(msgs, 300).splitlines():
        q = re.sub(r"^[\s\-\d\.\)۔*•]+", "", line).strip()
        if 8 < len(q) < 160 and URDU.match(q) and q.endswith(("؟", "?")):
            out.append(q)
    return list(dict.fromkeys(out))
gen, thin = [], 0
for i, c in enumerate(chunks):
    qs = questions(c["text"])
    if len(qs) < 3:
        thin += 1
        qs = list(dict.fromkeys(qs + questions(c["text"], strict=True)))
    gen += [{"q": q, "chunk": i} for q in qs]
del llm; gc.collect(); torch.cuda.empty_cache()
leaks = sum(g["q"] in {r["question"].strip() for r in qa} for g in gen)
print(f"DR-RAG questions: {len(gen)} for {N} chunks, retried {thin} chunks, identical to ours {leaks}")

# 3. Retrieval
FOLD = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ە": "ہ", "ۀ": "ۂ"})
tok = lambda s: re.findall(r"\w+", unicodedata.normalize("NFKC", s).translate(FOLD).lower())
bm = BM25Okapi([tok(c["text"]) for c in chunks])
enc = SentenceTransformer("BAAI/bge-m3", device="cuda"); enc.max_seq_length = 1024
rer = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=1024)
try: rer.model.half()
except Exception: pass
C = enc.encode([c["text"] for c in chunks], batch_size=8, normalize_embeddings=True)
G = enc.encode([g["q"] for g in gen], batch_size=32, normalize_embeddings=True)
gchunk = np.array([g["chunk"] for g in gen])
qtexts = sorted({r["question"] for r in qa} | set(translit.values()))
E = dict(zip(qtexts, enc.encode(qtexts, batch_size=32, normalize_embeddings=True)))

def rrf(*rows, k=60):
    f = np.zeros(N)
    for s in rows:
        for p, j in enumerate(np.argsort(-s)):
            f[j] += 1 / (k + p + 1)
    return f
cache = {}
def rerank(q, order):
    top = order[:20]
    need = [j for j in top if (q, j) not in cache]
    if need:
        for j, v in zip(need, rer.predict([(q, chunks[j]["text"]) for j in need], batch_size=8)):
            cache[(q, j)] = float(v)
    return sorted(top, key=lambda j: -cache[(q, j)]) + order[20:]
def ours(r):
    if r["lang"] == "en":
        return rerank(r["question"], [int(j) for j in np.argsort(-(C @ E[r["question"]]))])
    q = translit[r["id"]] if r["lang"] == "ur-Latn" else r["question"]
    return rerank(q, [int(j) for j in np.argsort(-rrf(bm.get_scores(tok(q)), C @ E[q]))])
def dr_rag(q, tau=0.80):
    s_qa = G @ E[q]
    by_chunk = np.full(N, -1.0)
    np.maximum.at(by_chunk, gchunk, s_qa)
    dense = [int(j) for j in np.argsort(-(C @ E[q]))]
    if s_qa.max() >= tau:
        first = [int(j) for j in np.argsort(-by_chunk) if by_chunk[j] > -1]
        return first + [j for j in dense if j not in first], True
    return dense, False

def score(rows, fn):
    out = {"r1": [], "r5": [], "mrr": []}; hits = 0
    for r in rows:
        rk, used = fn(r)
        hits += used
        gold = set(r["kb_chunk_ids"])
        out["r1"].append(float(ids[rk[0]] in gold))
        out["r5"].append(float(any(ids[j] in gold for j in rk[:5])))
        out["mrr"].append(next((1 / (p + 1) for p, j in enumerate(rk) if ids[j] in gold), 0.0))
    res = {k: round(float(np.mean(v)), 3) for k, v in out.items()}
    res["qa_index_share"] = round(hits / len(rows), 3)
    return res

METHODS = {
    "DR-RAG (paper)": lambda r: dr_rag(r["question"]),
    "DR-RAG + our reranker": lambda r: (rerank(r["question"], dr_rag(r["question"])[0]), dr_rag(r["question"])[1]),
    "DR-RAG + our script conversion + reranker": lambda r: (rerank(translit.get(r["id"], r["question"]), dr_rag(translit.get(r["id"], r["question"]))[0]), dr_rag(translit.get(r["id"], r["question"]))[1]),
    "Ours (routed + conversion + reranker)": lambda r: (ours(r), False),
}
result = {}
print(f"\nUnseen half (n=41 per language)\n{'lang':8} {'method':44} {'R@1':>6} {'R@5':>6} {'MRR':>6} {'QA idx':>7}")
for lang in ["ur", "ur-Latn", "en"]:
    rows = [r for r in test if r["lang"] == lang]
    result[lang] = {}
    for m, fn in METHODS.items():
        if lang == "en" and "conversion" in m and "Ours" not in m:
            continue
        v = score(rows, fn)
        result[lang][m] = v
        print(f"{lang:8} {m:44} {v['r1']:>6} {v['r5']:>6} {v['mrr']:>6} {v['qa_index_share']:>7}")
    print()

json.dump({"llm": name + " (4-bit, greedy)", "split": "test half of e3_heldout_kb_v2 (seed 42)",
           "conversion": {"n": len(translit), "retried": retried, "still_not_pure_urdu": still_bad},
           "dr_rag": {"paper": "Ahmad et al. 2026, CHiPSAL, ACL Anthology 2026.chipsal-1.6", "questions": len(gen),
                      "chunks_retried": thin, "identical_to_ours": leaks, "tau": 0.80},
           "results": result, "transliterations": translit, "qa_index": gen},
          open("/kaggle/working/e3_final_kb_v2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("saved /kaggle/working/e3_final_kb_v2.json")
